In [2]:
from clip_gnn.dataset import ClipDataset
from clip_gnn.model import ClipGPS
from clip_gnn.evaluate import evaluate
from torch_geometric.loader import DataLoader
import torch
import glob

/home/s2850039/miniconda3/envs/clip_gnn_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
device = torch.device("cuda")

Ssd1 = ClipDataset("datasets/ssd1_100nt_clip.parquet")
Ssd1_300 = ClipDataset("datasets/ssd1_300nt_clip.parquet")
Nab3 = ClipDataset("datasets/nab3_100nt_clip.parquet")
Nab3_50 = ClipDataset("datasets/nab3_50nt_clip.parquet")

def evaluate_test(ckpt_path, test_loader, protein):
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=True)
    model = ClipGPS(
        node_in_dim=ckpt["hyperparams"]["node_in_dim"],
        edge_in_dim=ckpt["hyperparams"]["edge_in_dim"],
        hidden_dim=ckpt["hyperparams"]["hidden_dim"],
        num_layers=ckpt["hyperparams"]["num_layers"],
        num_heads=ckpt["hyperparams"]["num_heads"],
        dropout=ckpt["hyperparams"]["dropout"],
        bilstm_layers=ckpt["hyperparams"]["bilstm_layers"],
    ).to(device)
    model.load_state_dict(ckpt["model_state_dict"])
    return evaluate(model, test_loader, device, protein)



[dataset] Loading graph cache from ssd1_custom_intervals_vienna/ssd1_clip.graphs.pt …
[dataset] Loaded 28,537 cached graphs.
[dataset] Loading graph cache from ssd1_custom_intervals_vienna/ssd1_clip_300.graphs.pt …
[dataset] Loaded 13,785 cached graphs.
[dataset] Loading graph cache from nab3_custom_intervals_vienna/nab3_clip100.graphs.pt …
[dataset] Loaded 15,841 cached graphs.
[dataset] Loading graph cache from nab3_custom_intervals_vienna/nab3_clip.graphs.pt …
[dataset] Loaded 18,290 cached graphs.


In [ ]:

results = {}

# main + BLSTM checkpoints (100nt data)
for data, protein in [(Ssd1, "ssd1"), (Nab3, "nab3")]:
    _, _, test_ds = data.split(val_frac=0.15, test_frac=0.15, seed=42)
    test_loader = DataLoader(test_ds, batch_size=16, shuffle=False)

    matches = glob.glob(f"{protein}_fit/tune_pass2/*_chosen/best_model.pt")
    results[f"{protein}_main"] = evaluate_test(matches[0], test_loader, protein)

    if protein == "ssd1":
        matches2 = glob.glob(f"{protein}_fit_blstm/tune_pass2/*_chosen/best_model.pt")
        results[f"{protein}_blstm"] = evaluate_test(matches2[0], test_loader, protein)

# fit50 (Nab3_50) and fit300/blstm_300nt (Ssd1_300) checkpoints
_, _, nab3_50_test = Nab3_50.split(val_frac=0.15, test_frac=0.15, seed=42)
nab3_50_loader = DataLoader(nab3_50_test, batch_size=16, shuffle=False)
matches = glob.glob("nab3_50nt_fit/tune_pass2/*_chosen/best_model.pt")
results["nab3_50nt_fit"] = evaluate_test(matches[0], nab3_50_loader, "Nab3")

_, _, ssd1_300_test = Ssd1_300.split(val_frac=0.15, test_frac=0.15, seed=42)
ssd1_300_loader = DataLoader(ssd1_300_test, batch_size=16, shuffle=False)
matches1 = glob.glob("results_custom/ssd1_300nt_fit/tune_pass2/*_chosen/best_model.pt")
results["ssd1_300nt_fit"] = evaluate_test(matches1[0], ssd1_300_loader, "Ssd1")

for name, metrics in results.items():
    print(name, metrics)

ssd1_main {'ssd1_auroc': 0.6192862314739257, 'ssd1_aupr': 0.6453350871260086, 'ssd1_f1': 0.6901864119545615, 'ssd1_f1_threshold': 0.11568234115839005, 'ssd1_brier': 0.2544950842857361, 'ssd1_ece': 0.11258751979080317, 'ssd1_mcc': 0.1594525587158959}
ssd1_blstm {'ssd1_auroc': 0.6487238900886063, 'ssd1_aupr': 0.6717974433658547, 'ssd1_f1': 0.691645569619782, 'ssd1_f1_threshold': 0.275246262550354, 'ssd1_brier': 0.2367817461490631, 'ssd1_ece': 0.058088122945383326, 'ssd1_mcc': 0.2162563126157442}
nab3_main {'nab3_auroc': 0.8751497590716536, 'nab3_aupr': 0.8894366585363129, 'nab3_f1': 0.8132281097326851, 'nab3_f1_threshold': 0.38665565848350525, 'nab3_brier': 0.14508815109729767, 'nab3_ece': 0.04520707130306699, 'nab3_mcc': 0.5930208503990656}
Nab3_fit50 {'Nab3_auroc': 0.863798049340218, 'Nab3_aupr': 0.8825281091368278, 'Nab3_f1': 0.820279493012175, 'Nab3_f1_threshold': 0.3306237459182739, 'Nab3_brier': 0.15561391413211823, 'Nab3_ece': 0.084935195534889, 'Nab3_mcc': 0.5803919067382874}
ssd